In [1]:
# Import the libraries we need
import pandas as pd        # pandas: handles data in table form (like a spreadsheet)
import sqlite3             # sqlite3: lets Python read our database file
import numpy as np         # numpy: does fast math (we'll need it for distance/angle soon)
import ast                 # ast: turns text like "[110.3, 43.2]" back into a real list

# Open the database file we saved back in Phase 1
# The 'r' makes it a "raw string" so the backslashes in the Windows path don't break
conn = sqlite3.connect(r'C:\Users\User\Desktop\football-xg-model\data\football_xg.db')

# Ask the database for all the shots and load them into a table called shots_df
# "SELECT * FROM shots" = "give me everything from the shots table"
shots_df = pd.read_sql("SELECT * FROM shots", conn)

# Confirm it worked by printing how many shots came back (should say 9168)
print("Loaded", shots_df.shape[0], "shots from the database")

# Show the first 5 rows so we can see what the data looks like
shots_df.head()

Loaded 9168 shots from the database


,id,match_id,player,team,shot_statsbomb_xg,shot_body_part,shot_technique,shot_type,shot_outcome,location,goal
0,a1501eed-1c34-4060-a92a-6bb8cef36c39,3825739,Jonathan Rodríguez Menéndez,Sporting Gijón,0.125582,Left Foot,Normal,Open Play,Blocked,"[102.3, 49.3]",0
1,085c74bd-911f-4e55-965d-4c7c94088a67,3825739,Gareth Frank Bale,Real Madrid,0.245234,Head,Normal,Open Play,Goal,"[116.8, 44.5]",1
2,10c2092d-0869-4f06-b6d7-e4c2bb99a07b,3825739,Cristiano Ronaldo dos Santos Aveiro,Real Madrid,0.060526,Left Foot,Half Volley,Open Play,Goal,"[103.6, 32.2]",1
3,8f9dac21-9fe8-4f59-be34-87cd48b1c057,3825739,Luka Modrić,Real Madrid,0.024467,Right Foot,Normal,Open Play,Blocked,"[90.7, 43.1]",0
4,4ad739ee-1133-491f-9760-b19bb6d407e2,3825739,Karim Benzema,Real Madrid,0.088242,Right Foot,Overhead Kick,Open Play,Goal,"[109.1, 38.4]",1


In [2]:
# The 'location' column is stored as TEXT like "[110.3, 43.2]" (because SQLite can't store lists)
# ast.literal_eval safely converts that text back into a real Python list [110.3, 43.2]
shots_df['location_parsed'] = shots_df['location'].apply(
    lambda loc: ast.literal_eval(loc) if isinstance(loc, str) else loc
)

# Take the FIRST number from each location (the x coordinate) and put it in its own column
# x = how far along the pitch, from 0 (own goal line) to 120 (the goal we're attacking)
shots_df['x'] = shots_df['location_parsed'].apply(lambda loc: loc[0])

# Take the SECOND number (the y coordinate) and put it in its own column
# y = how far across the pitch, from 0 (one sideline) to 80 (the other sideline), 40 = dead center
shots_df['y'] = shots_df['location_parsed'].apply(lambda loc: loc[1])

# Show the new x and y columns next to whether it was a goal, to check it worked
shots_df[['x', 'y', 'goal']].head()

,x,y,goal
0,102.3,49.3,0
1,116.8,44.5,1
2,103.6,32.2,1
3,90.7,43.1,0
4,109.1,38.4,1


In [3]:
# ---- DISTANCE TO GOAL ----
# The goal we attack is at x=120, and its center is at y=40
# We measure distance from each shot to that goal center point (120, 40)

# How far the shot is from the goal line along the pitch (horizontal gap)
# Example: a shot at x=110 is 120-110 = 10 units from the goal line
x_dist = 120 - shots_df['x']

# How far the shot is from the center line across the pitch (sideways gap)
# abs() makes it positive whether the shot is left OR right of center
# Example: a shot at y=45 is |40-45| = 5 units off-center
y_dist = (shots_df['y'] - 40).abs()

# Pythagoras: straight-line distance = square root of (x_dist² + y_dist²)
# np.sqrt = square root, ** 2 = squared. This gives the true distance to goal center.
shots_df['distance'] = np.sqrt(x_dist**2 + y_dist**2)


# ---- ANGLE TO GOAL ----
# The goal is 8 yards wide: posts sit at y=36 and y=44 (so the goal spans 8 units)
# The "angle" is how wide those two posts look from where the shooter stands.
# Wider angle = more open goal to aim at = easier.

# We use a trig formula. It measures the angle between the lines from the shot
# to each of the two goalposts. np.arctan2 safely handles the geometry for us.
goal_width = 8  # the goal is 8 units wide (from post at y=36 to post at y=44)

# This is the standard xG angle formula. It computes the shooting angle in radians.
# The top part relates to the goal width; the bottom relates to the shot's position.
angle = np.arctan2(
    goal_width * x_dist,                                  # numerator: goal width scaled by distance from goal line
    x_dist**2 + y_dist**2 - (goal_width / 2)**2           # denominator: geometry of the shot's position
)

# arctan2 can return negative angles; if it does, add pi (180°) to keep it positive
shots_df['angle'] = np.where(angle < 0, angle + np.pi, angle)

# Show our two new features next to whether it was a goal, to check they look sensible
shots_df[['x', 'y', 'distance', 'angle', 'goal']].head()

,x,y,distance,angle,goal
0,102.3,49.3,19.994499,0.353466,0
1,116.8,44.5,5.521775,1.055740,1
2,103.6,32.2,18.160396,0.396012,1
3,90.7,43.1,29.463537,0.268445,0
4,109.1,38.4,11.016805,0.691321,1


In [4]:
# Let's check: do goals really happen at shorter distances than misses?
# We group the shots by whether they were a goal (1) or not (0),
# then look at the AVERAGE distance for each group.

# .groupby('goal') splits shots into two groups: misses (0) and goals (1)
# ['distance'].mean() calculates the average distance within each group
avg_distance = shots_df.groupby('goal')['distance'].mean()

# Print the result. We EXPECT goals (group 1) to have a smaller average distance.
print("Average distance to goal:")
print(avg_distance)
print()

# Do the same for angle — we expect goals to have a WIDER average angle
avg_angle = shots_df.groupby('goal')['angle'].mean()
print("Average shooting angle:")
print(avg_angle)

Average distance to goal:
goal
0    19.864623
1    12.465677
Name: distance, dtype: float64

Average shooting angle:
goal
0    0.411866
1    0.694028
Name: angle, dtype: float64


In [5]:
# ---- EXTRA FEATURES from columns we already have ----

# 1) HEADER or not
# Headers are harder to score than shots with the foot, so this matters.
# We check the 'shot_body_part' column: if it says "Head", mark 1, otherwise 0.
# .astype(int) turns True/False into 1/0 so the model can use it.
shots_df['is_header'] = (shots_df['shot_body_part'] == 'Head').astype(int)

# 2) PENALTY or not
# Penalties are scored ~76% of the time — a totally different animal from open play.
# The 'shot_type' column tells us; if it's "Penalty", mark 1, otherwise 0.
shots_df['is_penalty'] = (shots_df['shot_type'] == 'Penalty').astype(int)

# 3) OPEN PLAY or not
# Shots from open play behave differently from free kicks/corners etc.
# If 'shot_type' is "Open Play", mark 1, otherwise 0.
shots_df['is_open_play'] = (shots_df['shot_type'] == 'Open Play').astype(int)

# Let's see our full feature set now, next to the goal column
# These are the columns we'll eventually feed to the model.
shots_df[['distance', 'angle', 'is_header', 'is_penalty', 'is_open_play', 'goal']].head(10)

,distance,angle,is_header,is_penalty,is_open_play,goal
0,19.994499,0.353466,0,0,1,0
1,5.521775,1.055740,1,0,1,1
2,18.160396,0.396012,0,0,1,1
3,29.463537,0.268445,0,0,1,0
4,11.016805,0.691321,0,0,1,1
5,18.103315,0.428499,0,0,1,0
6,10.214206,0.720822,0,0,1,1
7,25.401181,0.308909,0,0,1,0
8,21.633308,0.126750,0,0,1,0
9,12.841339,0.521746,0,0,1,0


In [6]:
# Sanity check: penalties should score WAY more often than normal shots.
# If our is_penalty flag is correct, the goal rate for penalties should be very high (~75%).

# .groupby('is_penalty') splits shots into two groups: normal shots (0) and penalties (1)
# ['goal'].mean() gives the fraction that were goals in each group (the goal rate)
penalty_goal_rate = shots_df.groupby('is_penalty')['goal'].mean()

# Print it as percentages so it's easy to read
print("Goal rate by penalty vs non-penalty:")
print((penalty_goal_rate * 100).round(1))
print()

# Also count how many penalties we have, just to know the sample size
# .sum() adds up all the 1s in is_penalty, giving the total number of penalties
print("Number of penalties in the data:", shots_df['is_penalty'].sum())

Goal rate by penalty vs non-penalty:
is_penalty
0    10.4
1    71.1
Name: goal, dtype: float64

Number of penalties in the data: 97


In [7]:
# We'll save our shots WITH the new features into a fresh table in the same database.
# This way Phase 3 can load ready-to-use features instead of recalculating them.

# Pick the columns we want to keep: our features + the target (goal) + some ID info
# (keeping player/team helps later for the player analysis phase)
columns_to_keep = ['id', 'match_id', 'player', 'team',
                   'distance', 'angle', 'is_header', 'is_penalty', 'is_open_play',
                   'shot_statsbomb_xg', 'goal']

# Make a clean copy with just those columns
features_df = shots_df[columns_to_keep].copy()

# Save it to the database as a NEW table called 'shot_features'
# if_exists='replace' means: if this table already exists, overwrite it
# index=False means: don't save pandas' row numbers as a column
features_df.to_sql('shot_features', conn, if_exists='replace', index=False)

# Confirm it saved by counting the rows in the new table using SQL
check = pd.read_sql("SELECT COUNT(*) AS rows FROM shot_features", conn)
print("Saved 'shot_features' table with", check['rows'][0], "rows")

# Peek at the final feature table
features_df.head()

Saved 'shot_features' table with 9168 rows


,id,match_id,player,team,distance,angle,is_header,is_penalty,is_open_play,shot_statsbomb_xg,goal
0,a1501eed-1c34-4060-a92a-6bb8cef36c39,3825739,Jonathan Rodríguez Menéndez,Sporting Gijón,19.994499,0.353466,0,0,1,0.125582,0
1,085c74bd-911f-4e55-965d-4c7c94088a67,3825739,Gareth Frank Bale,Real Madrid,5.521775,1.055740,1,0,1,0.245234,1
2,10c2092d-0869-4f06-b6d7-e4c2bb99a07b,3825739,Cristiano Ronaldo dos Santos Aveiro,Real Madrid,18.160396,0.396012,0,0,1,0.060526,1
3,8f9dac21-9fe8-4f59-be34-87cd48b1c057,3825739,Luka Modrić,Real Madrid,29.463537,0.268445,0,0,1,0.024467,0
4,4ad739ee-1133-491f-9760-b19bb6d407e2,3825739,Karim Benzema,Real Madrid,11.016805,0.691321,0,0,1,0.088242,1
